In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, 
                           f1_score, accuracy_score, precision_score, 
                           precision_recall_curve, recall_score, confusion_matrix)
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline

# Load the dataset
data = pd.read_excel(# enter file path)

# Prepare data with LASSO feature selection
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome
feature_indexes = [24, 38, 20, 16, 15, 12, 3, 10, 13, 21, 9, 22, 26, 33, 19, 2, 35, 32, 28, 6, 30]  # LASSO-selected features
X = X.iloc[:, feature_indexes]

# Define Logistic Regression with specified hyperparameters
classifier = LogisticRegression(
    C=30,
    penalty='l2',
    solver='sag',
    max_iter=1000,
    random_state=42
)

# Define pipeline with RandomOverSampler
pipeline = Pipeline([
    ('resampler', RandomOverSampler(sampling_strategy='auto', random_state=42)),
    ('classifier', classifier)
])

# Initialize cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Storage variables
all_actuals = []
all_probs = []
auc_scores = []
auprc_scores = []

# Cross-validation loop
for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]  # Probability of positive class
    
    # Store results for combined threshold calculation
    all_actuals.extend(y_test.values)
    all_probs.extend(y_prob)
    
    # Calculate fold metrics
    auc_scores.append(roc_auc_score(y_test, y_prob))
    auprc_scores.append(average_precision_score(y_test, y_prob))

# Find optimal threshold using all predictions
precision, recall, thresholds = precision_recall_curve(all_actuals, all_probs)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

# Recalculate metrics with optimal threshold
f1_list, acc_list, prec_list, sens_list, spec_list = [], [], [], [], []
for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= best_threshold).astype(int)
    
    # Compute metrics
    f1_list.append(f1_score(y_test, y_pred))
    acc_list.append(accuracy_score(y_test, y_pred))
    prec_list.append(precision_score(y_test, y_pred, zero_division=0))
    sens_list.append(recall_score(y_test, y_pred))  # Sensitivity = Recall
    
    # Calculate specificity (TN / (TN + FP))
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp + 1e-9)  # Avoid division by zero
    spec_list.append(specificity)

# Calculate statistics
def format_metric(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

print("=== Logistic Regression with RandomOverSampler ===")
print(f"Optimal Threshold (F1-maximizing): {best_threshold:.4f}")
print(f"Average AUC: {format_metric(np.mean(auc_scores), np.std(auc_scores))}")
print(f"Average AUPRC: {format_metric(np.mean(auprc_scores), np.std(auprc_scores))}")
print(f"F1 Score: {format_metric(np.mean(f1_list), np.std(f1_list))}")
print(f"Accuracy: {format_metric(np.mean(acc_list), np.std(acc_list))}")
print(f"Precision: {format_metric(np.mean(prec_list), np.std(prec_list))}")
print(f"Sensitivity (Recall): {format_metric(np.mean(sens_list), np.std(sens_list))}")
print(f"Specificity: {format_metric(np.mean(spec_list), np.std(spec_list))}")

=== Logistic Regression with RandomOverSampler ===
Optimal Threshold (F1-maximizing): 0.6390
Average AUC: 0.6693 ± 0.0462
Average AUPRC: 0.2031 ± 0.0460
F1 Score: 0.2764 ± 0.0603
Accuracy: 0.8338 ± 0.0169
Precision: 0.2303 ± 0.0512
Sensitivity (Recall): 0.3474 ± 0.0783
Specificity: 0.8827 ± 0.0143
